In [1]:
%load_ext cudf.pandas
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import time
import cupy as cp

In [2]:
data = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv')
scout = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv')
plays = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv')
players = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv')

# Let's merge these data into one set 
data = data.merge(scout, how='left')

In [3]:
data = pd.concat([data]*factor)

In [4]:
### cell 0
# get ball snap indicies 
_idxs = (data
         .loc[data['event']=='ball_snap', 
              'frameId']
         .index
         .values)

# to get 500ms of movement after snap, get 5 rows (each row is 100ms of info)
x = [(_idxs+x).tolist() for x in range(0,6)]
idxs = [item for sublist in x for item in sublist] #the output x is a list of lists, so this is just to flatten the list

# filter for snap + 500ms of data only using our selected indicies
_df = data.loc[idxs]

In [5]:
### cell 1
gid = 2021090900
pid = 97 
nid = 25511 
_ = _df.loc[(_df['gameId']==gid) & (_df['playId']==pid) & (_df['nflId']==nid)]

In [6]:
### cell 2
_los = (data
        .loc[(data['team']=='football') & 
             (data['frameId']==1), 
             ['gameId', 'playId', 'x']]
        .rename(columns={'x':'los'}))

# merge LOS info back to subsetted data
_df = _df.merge(_los)

In [7]:
### cell 3
_ = _df.loc[(_df['gameId']==gid) & (_df['playId']==pid) & (_df['nflId']==nid)]

In [8]:
%%time
### cell 4
# get difference from LOS for all frames and players 
_df['los_diff'] = _df['x'].sub(_df['los'])

# multiply by -1 for plays going the "left" direction 
# this is so pass block is monotonic in the same direction (as well as pass rush)
_df.loc[_df['playDirection']=='left', 'los_diff'] = _df.loc[_df['playDirection']=='left', 'los_diff'].mul(-1)

# merge onto play info to get possession team (could do this anywhere, i do it here for no real optimal reason)
_df = plays.loc[:, ['gameId', 'playId', 'possessionTeam']].merge(_df)

                                                                                                                  
                                            Total time elapsed: 0.436 seconds                                     
                                          13 GPU function calls in 0.157 seconds                                  
                                          0 CPU function calls in 0.000 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__        │ 4          │ 0.008       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ Series.sub                   │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__        │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__eq__              │ 2          │ 0.005       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__getitem__ │ 2          │ 0.012       │ 0.006       │ 0          │ 0.000       │ 0.000       │
│ Series.mul                   │ 1          │ 0.016       │ 0.016       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__setitem__ │ 1          │ 0.074       │ 0.074       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.merge              │ 1          │ 0.028       │ 0.028       │ 0          │ 0.000       │ 0.000       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘